# Part 3: SQL Analysis

## Goal
Use SQL on the cleaned Global Findex dataset to answer business-style question in  a reproducible way.

## What this notebook will do
- Load the cleaned dataset  into SQLITE
- Validate the SQL table
- Analyze overall coverage
- Start Country-level analysis with Nepal

## Imports

In [1]:
import sqlite3
from pathlib import Path 
import pandas as pd

In [2]:
project_path = Path(r"D:\financial-inclusion-gap-analysis")
csv_path = project_path / "data" / "processed" / "findex_analysis_ready.csv"
db_path = project_path / "data" / "processed" / "findex_analysis.db"

print(csv_path)
print(db_path)
print(csv_path.exists(), db_path.exists())

D:\financial-inclusion-gap-analysis\data\processed\findex_analysis_ready.csv
D:\financial-inclusion-gap-analysis\data\processed\findex_analysis.db
True True


## Load the cleaned dataset into SQLite
We use SQLite so we can practice SQl on the processed dataset without depending on external database software

In [3]:
df = pd.read_csv(csv_path)
df.shape

(8577, 18)

In [4]:
conn = sqlite3.connect(db_path)
df.to_sql("findex_clean", conn, if_exists="replace", index=False)
print("Table loaded to SQLite")

Table loaded to SQLite


## First SQL Checks
These checks confirms that the SQL tabel matches the cleaned dataset.

In [5]:
query = "SELECT COUNT(*) AS total_rows FROM findex_clean"
pd.read_sql_query(query, conn)

,total_rows
0,8577


In [6]:
query = """
SELECT entity_type, COUNT(*) AS row_count
FROM findex_clean
GROUP BY entity_type
ORDER BY row_count DESC;
"""
pd.read_sql(query, conn)

,entity_type,row_count
0,Country,7893
1,Aggregate,684


In [7]:
query = """
SELECT year, COUNT(*) AS row_count
FROM findex_clean
GROUP BY year
ORDER BY year
"""
pd.read_sql(query, conn)

,year,row_count
0,2011,1516
1,2014,1686
2,2017,1734
3,2021,1483
4,2022,176
5,2024,1982


In [8]:
query = """
SELECT COUNT(DISTINCT country_code) AS distinct_country_codes
FROM findex_clean
"""
pd.read_sql(query, conn)

,distinct_country_codes
0,174


## Nepal Analysis
We now filter to Nepal and inspect financial inclusion indicators over time.

In [9]:
query = """
SELECT *
FROM findex_clean
WHERE country_code = 'NPL'
LIMIT 10
"""
pd.read_sql(query, conn)

,country,country_code,entity_type,year,survey_period_type,adult_population,region,income_group,group,group2,account_ownership,financial_institution_account,mobile_money_account,digital_access,digital_merchant_payment,digital_payment_made,digital_payment_received,inactive_account
0,Nepal,NPL,Country,2011,Standard survey wave,17498633.0,South Asia (excluding high income),Lower middle income,all,all,0.253086,0.253086,NaN,NaN,NaN,NaN,NaN,NaN
1,Nepal,NPL,Country,2014,Standard survey wave,18161615.0,South Asia (excluding high income),Lower middle income,all,all,0.338014,0.338014,0.003360,NaN,NaN,0.056214,0.065320,0.057346
2,Nepal,NPL,Country,2017,Standard survey wave,18869412.0,South Asia (excluding high income),Lower middle income,all,all,0.453856,0.453856,NaN,NaN,NaN,0.091124,0.110819,0.112355
3,Nepal,NPL,Country,2021,Standard survey wave,20254590.0,South Asia (excluding high income),Lower middle income,all,all,0.540027,0.528021,0.060919,NaN,0.054149,0.187359,0.194268,0.140585
4,Nepal,NPL,Country,2024,Standard survey wave,21168213.0,South Asia (excluding high income),Lower middle income,all,all,0.599911,0.575257,0.105834,0.176216,0.092866,0.170105,0.212262,0.106261
5,Nepal,NPL,Country,2011,Standard survey wave,17498633.0,South Asia (excluding high income),Lower middle income,gender,men,0.295582,0.295582,NaN,NaN,NaN,NaN,NaN,NaN
6,Nepal,NPL,Country,2011,Standard survey wave,17498633.0,South Asia (excluding high income),Lower middle income,gender,women,0.212196,0.212196,NaN,NaN,NaN,NaN,NaN,NaN
7,Nepal,NPL,Country,2014,Standard survey wave,18161615.0,South Asia (excluding high income),Lower middle income,gender,men,0.367245,0.367245,NaN,NaN,NaN,NaN,NaN,NaN
8,Nepal,NPL,Country,2014,Standard survey wave,18161615.0,South Asia (excluding high income),Lower middle income,gender,women,0.312665,0.312665,NaN,NaN,NaN,NaN,NaN,NaN
9,Nepal,NPL,Country,2017,Standard survey wave,18869412.0,South Asia (excluding high income),Lower middle income,gender,men,0.499813,0.499813,NaN,NaN,NaN,NaN,0.142650,0.136624


In [10]:
query = """
SELECT
    year,
    account_ownership,
    digital_access,
    mobile_money_account,
    inactive_account
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" = 'all'
    AND "group2" = 'all'
ORDER BY year
"""
pd.read_sql(query, conn)

,year,account_ownership,digital_access,mobile_money_account,inactive_account
0,2011,0.253086,NaN,NaN,NaN
1,2014,0.338014,NaN,0.003360,0.057346
2,2017,0.453856,NaN,NaN,0.112355
3,2021,0.540027,NaN,0.060919,0.140585
4,2024,0.599911,0.176216,0.105834,0.106261


## Summary

Initial SQL validation confirms that the cleaned dataset was loaded correctly into SQLite.

The first Nepal-focused query filters to:
- `country_code = 'NPL'`
- `entity_type = 'Country'`
- `"group" = 'all'`
- `"group2" = 'all'`

This gives the national overall view for Nepal across survey years and avoids mixing subgroup rows into the trend.

## Nepal Trend Analysis
We now focus on specific indicators over time so the sql analysis starts producing interpretable country insights.

In [11]:
query = """
SELECT
    year,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" = 'all'
    AND "group2" = 'all'
"""
pd.read_sql(query, conn)

,year,account_ownership_pct
0,2011,25.31
1,2014,33.80
2,2017,45.39
3,2021,54.00
4,2024,59.99


In [12]:
query = """
SELECT
    year,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" = 'all'
    AND "group2" = 'all'
"""
pd.read_sql(query, conn)

,year,digital_access_pct
0,2011,NaN
1,2014,NaN
2,2017,NaN
3,2021,NaN
4,2024,17.62


In [13]:
query = """
SELECT
    year,
    ROUND(mobile_money_account * 100, 2) AS mobile_money_account_pct,
    ROUND(inactive_account * 100, 2) AS inactive_account_pct
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" = 'all'
    AND "group2" = 'all'
ORDER BY year
"""
pd.read_sql(query, conn)

,year,mobile_money_account_pct,inactive_account_pct
0,2011,NaN,NaN
1,2014,0.34,5.73
2,2017,NaN,11.24
3,2021,6.09,14.06
4,2024,10.58,10.63


## Benchmark Analysis
The next step is to compare Nepal with broader benchmarks so we can judge whether Nepal is underperforming, matching, or outperforming its peer groups.

In [14]:
query = """
SELECT DISTINCT region
FROM findex_clean
WHERE country_code = 'NPL'
ORDER BY region
"""
pd.read_sql(query,conn)

,region
0,South Asia (excluding high income)


In [15]:
query = """
SELECT DISTINCT income_group
FROM findex_clean
WHERE country_code = 'NPL'
ORDER BY income_group
"""
pd.read_sql(query, conn)

,income_group
0,Lower middle income


In [16]:
query = """
SELECT
    year,
    country,
    entity_type,
    region,
    income_group,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE "group" = 'all'
    AND "group2" = 'all'
    AND (
        country_code = 'NPL'
        OR country IN ('South Asia')
    )
ORDER BY year, country
"""
pd.read_sql(query, conn)

,year,country,entity_type,region,income_group,account_ownership_pct,digital_access_pct
0,2011,Nepal,Country,South Asia (excluding high income),Lower middle income,25.31,NaN
1,2011,South Asia,Aggregate,None,None,32.20,NaN
2,2014,Nepal,Country,South Asia (excluding high income),Lower middle income,33.80,NaN
3,2014,South Asia,Aggregate,None,None,46.32,NaN
4,2017,Nepal,Country,South Asia (excluding high income),Lower middle income,45.39,NaN
5,2017,South Asia,Aggregate,None,None,69.41,NaN
6,2021,Nepal,Country,South Asia (excluding high income),Lower middle income,54.00,NaN
7,2021,South Asia,Aggregate,None,None,68.00,NaN
8,2024,Nepal,Country,South Asia (excluding high income),Lower middle income,59.99,17.62
9,2024,South Asia,Aggregate,None,None,77.57,28.93


In [17]:
query = """
SELECT
    year,
    country,
    entity_type,
    income_group,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE "group" = 'all'
    AND "group2" = 'all'
    AND (
        country_code = 'NPL'
        OR income_group = 'Lower middle income'
    )
ORDER BY year, entity_type, country
LIMIT 30
"""
pd.read_sql(query, conn)

,year,country,entity_type,income_group,account_ownership_pct,digital_access_pct
0,2011,Algeria,Country,Lower middle income,33.29,None
1,2011,Angola,Country,Lower middle income,39.20,None
2,2011,Bangladesh,Country,Lower middle income,31.74,None
3,2011,Benin,Country,Lower middle income,10.46,None
4,2011,Bolivia,Country,Lower middle income,28.03,None
5,2011,Cambodia,Country,Lower middle income,3.66,None
6,2011,Cameroon,Country,Lower middle income,14.81,None
7,2011,Comoros,Country,Lower middle income,21.69,None
8,2011,"Congo, Rep.",Country,Lower middle income,10.05,None
9,2011,Djibouti,Country,Lower middle income,12.27,None


In [18]:
query = """
SELECT
    year,
    country,
    country_code,
    entity_type,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE "group" = 'all'
    AND "group2" = 'all'
    AND (
        country_code = 'NPL'
        OR country_code = 'SAS'
    )
ORDER BY year, country_code
"""
pd.read_sql(query, conn)

,year,country,country_code,entity_type,account_ownership_pct,digital_access_pct
0,2011,Nepal,NPL,Country,25.31,NaN
1,2011,South Asia,SAS,Aggregate,32.20,NaN
2,2014,Nepal,NPL,Country,33.80,NaN
3,2014,South Asia,SAS,Aggregate,46.32,NaN
4,2017,Nepal,NPL,Country,45.39,NaN
5,2017,South Asia,SAS,Aggregate,69.41,NaN
6,2021,Nepal,NPL,Country,54.00,NaN
7,2021,South Asia,SAS,Aggregate,68.00,NaN
8,2024,Nepal,NPL,Country,59.99,17.62
9,2024,South Asia,SAS,Aggregate,77.57,28.93


In [19]:
query = """
SELECT
    year,
    country,
    country_code,
    entity_type,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE "group" = 'all'
    AND "group2" = 'all'
    AND country_code IN ('NPL', 'SAS', 'WLD')
ORDER BY year, country_code
"""
pd.read_sql(query, conn)

,year,country,country_code,entity_type,account_ownership_pct,digital_access_pct
0,2011,Nepal,NPL,Country,25.31,NaN
1,2011,South Asia,SAS,Aggregate,32.20,NaN
2,2011,world,WLD,Aggregate,50.61,NaN
3,2014,Nepal,NPL,Country,33.80,NaN
4,2014,South Asia,SAS,Aggregate,46.32,NaN
5,2014,world,WLD,Aggregate,61.67,NaN
6,2017,Nepal,NPL,Country,45.39,NaN
7,2017,South Asia,SAS,Aggregate,69.41,NaN
8,2017,world,WLD,Aggregate,69.32,NaN
9,2021,Nepal,NPL,Country,54.00,NaN


In [20]:
query = """
SELECT
    year,
    country,
    country_code,
    entity_type,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE "group" = 'all'
    AND "group2" = 'all'
    AND country_code IN ('NPL', 'LMC')
ORDER BY year, country_code
"""
pd.read_sql(query, conn)

,year,country,country_code,entity_type,account_ownership_pct,digital_access_pct
0,2011,Lower middle income,LMC,Aggregate,30.45,NaN
1,2011,Nepal,NPL,Country,25.31,NaN
2,2014,Lower middle income,LMC,Aggregate,43.64,NaN
3,2014,Nepal,NPL,Country,33.80,NaN
4,2017,Lower middle income,LMC,Aggregate,60.01,NaN
5,2017,Nepal,NPL,Country,45.39,NaN
6,2021,Lower middle income,LMC,Aggregate,62.11,NaN
7,2021,Nepal,NPL,Country,54.00,NaN
8,2024,Lower middle income,LMC,Aggregate,70.39,36.82
9,2024,Nepal,NPL,Country,59.99,17.62


## Benchmark Interpretation

Use these benchmark queries to compare Nepal with:
- South Asia aggregate (`SAS`)
- World aggregate (`WLD`)
- Lower middle income aggregate (`LMC`)

Focus on whether Nepal is above or below each benchmark for:
- account ownership
- digital access

## Nepal Subgroup Gap Analysis

Benchmark comparision show Nepal again aggregate peers. The next step is to measure internal gap within Nepal by gender, age, and income group.

For long-term subgroup analysis, account ownership is the  most stable indicator across the survey years.

In [21]:
query = """
SELECT
    year,
    "group",
    "group2",
    ROUND(account_ownership * 100,2) AS account_ownership_pct
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" IN ('gender', 'age_cat', 'income')
ORDER BY year, "group", group2
"""
pd.read_sql(query, conn)

,year,group,group2,account_ownership_pct
0,2011,age_cat,age 25+,26.57
1,2011,age_cat,ages 15-24,23.65
2,2011,gender,men,29.56
3,2011,gender,women,21.22
4,2011,income,poorest 40%,14.44
5,2011,income,richest 60%,32.54
6,2014,age_cat,age 25+,37.34
7,2014,age_cat,ages 15-24,25.42
8,2014,gender,men,36.72
9,2014,gender,women,31.27


## Gender Gap in Nepal
This query compares account ownership between men and women in Nepal over time.

In [22]:
query = """
SELECT
    year,
    ROUND(MAX(CASE WHEN group2 = 'men' THEN account_ownership END) * 100, 2) AS men_pct,
    ROUND(MAX(CASE WHEN group2 = 'women' THEN account_ownership END) * 100, 2) AS women_pct,
    ROUND(
        (MAX(CASE WHEN group2 = 'men' THEN account_ownership END) -
        MAX(CASE WHEN group2 = 'women' THEN account_ownership END)) * 100,
        2
    ) AS gender_gap_pct_points
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" = 'gender'
GROUP BY year
ORDER by year
"""
pd.read_sql(query, conn)

,year,men_pct,women_pct,gender_gap_pct_points
0,2011,29.56,21.22,8.34
1,2014,36.72,31.27,5.46
2,2017,49.98,41.60,8.38
3,2021,58.61,49.90,8.71
4,2024,60.23,59.78,0.45


## Age Gap in Nepal
Ths comapares younger adults with older adults in Nepal.

In [23]:
query = """
SELECT
    year,
    ROUND(MAX(CASE WHEN group2 = 'ages 15-24' THEN account_ownership END) * 100,2) AS age_15_24_pct,
    ROUND(MAX(CASE WHEN group2 = 'age 25+' THEN account_ownership END) * 100, 2) AS age_25_plus_pct,
    ROUND(
    (MAX(CASE WHEN group2 = 'age 25+' THEN account_ownership END) -
    MAX(CASE WHEN group2 = 'ages 15-24' THEN account_ownership END)) * 100,
    2
    ) AS age_gap_pct_points
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" = 'age_cat'
GROUP BY year
ORDER BY year
"""
pd.read_sql(query, conn)

,year,age_15_24_pct,age_25_plus_pct,age_gap_pct_points
0,2011,23.65,26.57,2.92
1,2014,25.42,37.34,11.92
2,2017,39.34,47.72,8.39
3,2021,42.23,59.02,16.79
4,2024,39.12,67.37,28.26


## Income Gap in Nepal
This compares richer and poorer segments in Nepal

In [24]:
query = """
SELECT
    year,
    ROUND(MAX(CASE WHEN group2 = 'richest 60%' THEN account_ownership END) * 100,2) AS richest_60_pct,
    ROUND(MAX(CASE WHEN group2 = 'poorest 40%' THEN account_ownership END) * 100, 2) AS poorest_40_pct,
    ROUND(
    (MAX(CASE WHEN group2 = 'richest 60%' THEN account_ownership END) - 
    MAX(CASE WHEN group2 = 'poorest 40%' THEN account_ownership END)) * 100,
    2
    ) AS income_gap_pct_points
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" = 'income'
GROUP BY year
ORDER BY year
"""
pd.read_sql(query, conn)

,year,richest_60_pct,poorest_40_pct,income_gap_pct_points
0,2011,32.54,14.44,18.10
1,2014,40.21,24.13,16.07
2,2017,50.33,37.92,12.41
3,2021,60.28,44.55,15.73
4,2024,64.64,53.01,11.63


## Subgroup Interpretation

- Gender gaps narrows strongly and is almost closed by 2024.
- Age gap becomes the largest account-ownership gab by 2024.
- Income gap remains meaningful but improves over time.
- Nepal's inclusion growth is uneven across subgroups.

## 2024 Digital Access Subgroup Analysis

Digital access is not avaiable consistently across earlier survey years, so subgroup analysis should focus on 2024 rather than forcing a multi-year comparision with mostly missing values.

In [25]:
query = """
SELECT
    year,
    "group",
    group2,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND year = 2024
    AND "group" IN ('gender', 'age_cat', 'income')
ORDER BY "group", group2
"""
pd.read_sql(query, conn)

,year,group,group2,digital_access_pct
0,2024,age_cat,age 25+,15.13
1,2024,age_cat,ages 15-24,24.68
2,2024,gender,men,23.68
3,2024,gender,women,12.15
4,2024,income,poorest 40%,10.66
5,2024,income,richest 60%,22.26


## 2024 Digital Access Gender Gap

In [26]:
query = """
SELECT
    ROUND(MAX(CASE WHEN group2 = 'men' THEN digital_access END) * 100, 2) AS men_pct,
    ROUND(MAX(CASE WHEN group2 = 'women' THEN digital_access END) * 100, 2) AS women_pct,
    ROUND(
    (MAX(CASE WHEN group2 = 'men' THEN digital_access END) - 
    MAX(CASE WHEN group2 = 'women' THEN digital_access END)) * 100,
    2
    ) AS gender_gap_pct_points
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND year  = 2024
    AND "group" = 'gender'
"""
pd.read_sql(query, conn)

,men_pct,women_pct,gender_gap_pct_points
0,23.68,12.15,11.54


## 2024 Digital Access Age Gap

In [27]:
query = """
SELECT
    ROUND(MAX(CASE WHEN group2 = 'ages 15-24' THEN digital_access END) * 100,2) AS age_15_24_pct,
    ROUND(MAX(CASE WHEN group2 = 'age 25+' THEN digital_access END)* 100, 2) AS age_25_plus_pct,
    ROUND(
    (MAX(CASE WHEN group2 = 'age 25+'THEN digital_access END)-
    MAX(CASE WHEN group2 = 'ages 15-24' THEN digital_access END)) * 100,
    2
    ) AS age_gap_pct_points
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND year = 2024
    AND "group" = 'age_cat'
"""
pd.read_sql(query, conn)

,age_15_24_pct,age_25_plus_pct,age_gap_pct_points
0,24.68,15.13,-9.55


## Digital Access Income Gap

In [28]:
query = """
SELECT
    ROUND(MAX(CASE WHEN group2 = 'richest 60%' THEN digital_access END) * 100, 2) AS richest_60_pct,
    ROUND(MAX(CASE WHEN group2 = 'poorest 40%' THEN digital_access END) * 100, 2) AS poorest_40_pct,
    ROUND(
    (MAX(CASE WHEN group2 = 'richest 60%' THEN digital_access END) - 
    MAX(CASE WHEN group2 = 'poorest 40%' THEN digital_access END)) * 100,
    2
    ) AS income_gap_pct_points
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND year = 2024
    AND "group" = 'income'
"""
pd.read_sql(query, conn)

,richest_60_pct,poorest_40_pct,income_gap_pct_points
0,22.26,10.66,11.61


## 2024 Digital Access Interpretation

- Men have much higher digital access than women.
- Younger adult lead older adult in digital access.
- Richer groups have much higher digital access than poorer groups.
- 2024 digital inclusion gaps are sharper than account-ownership gaps.

## Export Power BI Summary Tables

Export compact summary tables for Power BI so the dashboards can be built from clean analytical outputs instead of repeating SQL logic inside Power BI.

In [29]:
benchmark_query = """
SELECT
    year,
    country,
    country_code,
    entity_type,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE "group" = 'all'
    AND "group2" = 'all'
    AND country_code IN ('NPL', 'SAS', 'LMC', 'WLD')
ORDER BY year, country_code
"""

nepal_benchmark_summary = pd.read_sql(benchmark_query, conn)
nepal_benchmark_summary

,year,country,country_code,entity_type,account_ownership_pct,digital_access_pct
0,2011,Lower middle income,LMC,Aggregate,30.45,NaN
1,2011,Nepal,NPL,Country,25.31,NaN
2,2011,South Asia,SAS,Aggregate,32.20,NaN
3,2011,world,WLD,Aggregate,50.61,NaN
4,2014,Lower middle income,LMC,Aggregate,43.64,NaN
5,2014,Nepal,NPL,Country,33.80,NaN
6,2014,South Asia,SAS,Aggregate,46.32,NaN
7,2014,world,WLD,Aggregate,61.67,NaN
8,2017,Lower middle income,LMC,Aggregate,60.01,NaN
9,2017,Nepal,NPL,Country,45.39,NaN


In [32]:
account_subgroup_query = """
SELECT
    year,
    "group",
    "group2",
    ROUND(account_ownership * 100, 2) AS account_ownership_pct
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" IN ('gender', 'age_cat', 'income')
ORDER  BY year, "group", "group2"
"""
nepal_account_subgroup_summary = pd.read_sql(account_subgroup_query, conn)
nepal_account_subgroup_summary

,year,group,group2,account_ownership_pct
0,2011,age_cat,age 25+,26.57
1,2011,age_cat,ages 15-24,23.65
2,2011,gender,men,29.56
3,2011,gender,women,21.22
4,2011,income,poorest 40%,14.44
5,2011,income,richest 60%,32.54
6,2014,age_cat,age 25+,37.34
7,2014,age_cat,ages 15-24,25.42
8,2014,gender,men,36.72
9,2014,gender,women,31.27


In [34]:
digital_2024_query = """
SELECT
    "group",
    group2,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND year = 2024
    AND "group" IN ('gender', 'agr_cat', 'income')
ORDER BY "group", group2
"""

nepal_digital_access_summary_2024 = pd.read_sql(digital_2024_query, conn)
nepal_digital_access_summary_2024

,group,group2,digital_access_pct
0,gender,men,23.68
1,gender,women,12.15
2,income,poorest 40%,10.66
3,income,richest 60%,22.26


In [37]:
output_dir = Path(r"D:/financial-inclusion-gap-analysis/data/processed")

nepal_benchmark_summary.to_csv(output_dir / "nepal_benchmark_summary.csv", index = False)
nepal_account_subgroup_summary.to_csv(output_dir / "nepal_account_subgroup_summary.csv", index = False)
nepal_digital_access_summary_2024.to_csv(output_dir / "nepal_digital_access_2024_summary.csv", index = False)

print("Tables Exported")

Tables Exported


In [39]:
for file_name in [
    "nepal_benchmark_summary.csv",
    "nepal_account_subgroup_summary.csv",
    "nepal_digital_access_2024_summary.csv"
]:
    path = output_dir / file_name
    print(file_name, path.exists(), path.stat().st_size)

nepal_benchmark_summary.csv True 836
nepal_account_subgroup_summary.csv True 884
nepal_digital_access_2024_summary.csv True 123
